In [1]:
!git clone https://github.com/EleutherAI/lm-evaluation-harness
%cd lm-evaluation-harness
!pip install -e .
!pip install accelerate bitsandbytes

Cloning into 'lm-evaluation-harness'...
remote: Enumerating objects: 65297, done.
remote: Total 65297 (delta 0), reused 0 (delta 0), pack-reused 65297 (from 1)
Receiving objects: 100% (65297/65297), 35.13 MiB | 17.37 MiB/s, done.
Resolving deltas: 100% (45146/45146), done.
/kaggle/working/lm-evaluation-harness
Obtaining file:///kaggle/working/lm-evaluation-harness
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Building editable for lm_eval (pyproject.toml) ... done
  Created wheel for lm_eval: filename=lm_eval-0.4.14.dev0-0.editable-py3-none-any.whl size=32319 

In [2]:
!pip install -q "transformers==4.57.1"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.26.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [3]:
%%writefile /kaggle/working/lm-evaluation-harness/lm_eval/tasks/hendrycks_math/utils.py
from typing import Dict, List

import datasets


def process_docs(dataset: datasets.Dataset) -> datasets.Dataset:
    def _process_doc(doc: dict) -> dict:
        out_doc = {
            "problem": doc["problem"],
            "solution": doc["solution"],
            "answer": remove_boxed(last_boxed_only_string(doc["solution"])),
        }
        return out_doc

    return dataset.map(_process_doc)


def process_results(doc: dict, results: List[str]) -> Dict[str, int]:
    retval = 0
    response = results[0]

    # Try to extract \boxed{} from the response (matches aime task behavior)
    boxed = last_boxed_only_string(response)
    if boxed is not None:
        try:
            answer = remove_boxed(boxed)
        except AssertionError:
            answer = None
    else:
        answer = None

    # Fall back to $...$ extraction
    if answer is None:
        indices = [pos for pos, char in enumerate(response) if char == "$"]
        if len(indices) <= 1:
            answer = response
        else:
            answer = response[indices[0] + 1 : indices[-1]]

    if is_equiv(answer, remove_boxed(last_boxed_only_string(doc["solution"]))):
        retval = 1

    results = {
        "exact_match": retval,
    }
    return results


# string normalization from https://github.com/EleutherAI/lm-evaluation-harness/blob/master/lm_eval/tasks/hendrycks_math.py
def is_equiv(str1, str2, verbose=False):
    if str1 is None and str2 is None:
        print("WARNING: Both None")
        return True
    if str1 is None or str2 is None:
        return False

    try:
        ss1 = strip_string(str1)
        ss2 = strip_string(str2)
        if verbose:
            print(ss1, ss2)
        return ss1 == ss2
    except Exception:
        return str1 == str2


def remove_boxed(s):
    if "\\boxed " in s:
        left = "\\boxed "
        assert s[: len(left)] == left
        return s[len(left) :]

    left = "\\boxed{"

    assert s[: len(left)] == left
    assert s[-1] == "}"

    return s[len(left) : -1]


def last_boxed_only_string(string):
    idx = string.rfind("\\boxed")
    if "\\boxed " in string:
        return "\\boxed " + string.split("\\boxed ")[-1].split("$")[0]
    if idx < 0:
        idx = string.rfind("\\fbox")
        if idx < 0:
            return None

    i = idx
    right_brace_idx = None
    num_left_braces_open = 0
    while i < len(string):
        if string[i] == "{":
            num_left_braces_open += 1
        if string[i] == "}":
            num_left_braces_open -= 1
            if num_left_braces_open == 0:
                right_brace_idx = i
                break
        i += 1

    if right_brace_idx is None:
        retval = None
    else:
        retval = string[idx : right_brace_idx + 1]

    return retval


def fix_fracs(string):
    substrs = string.split("\\frac")
    new_str = substrs[0]
    if len(substrs) > 1:
        substrs = substrs[1:]
        for substr in substrs:
            new_str += "\\frac"
            if substr[0] == "{":
                new_str += substr
            else:
                try:
                    assert len(substr) >= 2
                except AssertionError:
                    return string
                a = substr[0]
                b = substr[1]
                if b != "{":
                    if len(substr) > 2:
                        post_substr = substr[2:]
                        new_str += "{" + a + "}{" + b + "}" + post_substr
                    else:
                        new_str += "{" + a + "}{" + b + "}"
                else:
                    if len(substr) > 2:
                        post_substr = substr[2:]
                        new_str += "{" + a + "}" + b + post_substr
                    else:
                        new_str += "{" + a + "}" + b
    string = new_str
    return string


def fix_a_slash_b(string):
    if len(string.split("/")) != 2:
        return string
    a = string.split("/")[0]
    b = string.split("/")[1]
    try:
        a = int(a)
        b = int(b)
        assert string == "{}/{}".format(a, b)
        new_string = "\\frac{" + str(a) + "}{" + str(b) + "}"
        return new_string
    except AssertionError:
        return string


def remove_right_units(string):
    # "\\text{ " only ever occurs (at least in the val set) when describing units
    if "\\text{ " in string:
        splits = string.split("\\text{ ")
        assert len(splits) == 2
        return splits[0]
    else:
        return string


def fix_sqrt(string):
    if "\\sqrt" not in string:
        return string
    splits = string.split("\\sqrt")
    new_string = splits[0]
    for split in splits[1:]:
        if split[0] != "{":
            a = split[0]
            new_substr = "\\sqrt{" + a + "}" + split[1:]
        else:
            new_substr = "\\sqrt" + split
        new_string += new_substr
    return new_string


def strip_string(string):
    # linebreaks
    string = string.replace("\n", "")

    # remove inverse spaces
    string = string.replace("\\!", "")

    # replace \\ with \
    string = string.replace("\\\\", "\\")

    # replace tfrac and dfrac with frac
    string = string.replace("tfrac", "frac")
    string = string.replace("dfrac", "frac")

    # remove \left and \right
    string = string.replace("\\left", "")
    string = string.replace("\\right", "")

    # Remove circ (degrees)
    string = string.replace("^{\\circ}", "")
    string = string.replace("^\\circ", "")

    # remove dollar signs
    string = string.replace("\\$", "")

    # remove units (on the right)
    string = remove_right_units(string)

    # remove percentage
    string = string.replace("\\%", "")
    string = string.replace("\%", "")  # noqa: W605

    # " 0." equivalent to " ." and "{0." equivalent to "{." Alternatively, add "0" if "." is the start of the string
    string = string.replace(" .", " 0.")
    string = string.replace("{.", "{0.")
    # if empty, return empty string
    if len(string) == 0:
        return string
    if string[0] == ".":
        string = "0" + string

    # to consider: get rid of e.g. "k = " or "q = " at beginning
    if len(string.split("=")) == 2:
        if len(string.split("=")[0]) <= 2:
            string = string.split("=")[1]

    # fix sqrt3 --> sqrt{3}
    string = fix_sqrt(string)

    # remove spaces
    string = string.replace(" ", "")

    # \frac1b or \frac12 --> \frac{1}{b} and \frac{1}{2}, etc. Even works with \frac1{72} (but not \frac{72}1). Also does a/b --> \\frac{a}{b}
    string = fix_fracs(string)

    # manually change 0.5 --> \frac{1}{2}
    if string == "0.5":
        string = "\\frac{1}{2}"

    # NOTE: X/Y changed to \frac{X}{Y} in dataset, but in simple cases fix in case the model output is X/Y
    string = fix_a_slash_b(string)

    return string

Overwriting /kaggle/working/lm-evaluation-harness/lm_eval/tasks/hendrycks_math/utils.py


In [ ]:
!accelerate launch -m lm_eval \
    --model hf \
    --model_args pretrained=arefehRajabian/deepseek_r1_finetune_16bit_persian,parallelize=True,load_in_4bit=True \
    --tasks gsm8k \
    --device cuda:0 \
    --batch_size 1 \
    --limit 50 \
    --log_samples \
    --output_path /kaggle/working/results

In [5]:
!accelerate launch --num_processes=1 -m lm_eval --model hf \
    --tasks hendrycks_math_algebra hendrycks_math_counting_and_prob hendrycks_math_geometry hendrycks_math_intermediate_algebra hendrycks_math_num_theory hendrycks_math_prealgebra hendrycks_math_precalc \
    --model_args pretrained=arefehRajabian/deepseek_r1_finetune_16bit_persian,parallelize=True,load_in_4bit=True \
    --gen_kwargs m
    ax_new_tokens=1024 \
    --limit 5 \
    --log_samples \
    --batch_size 2 \
    --device cuda:0 \
    --output_path /content/deepseek \
    --system_instruction "Solve the following mathematics problem. Show your reasoning step by step. Use standard mathematical notation for fractions, radicals, equations, and other expressions. Put your final answer in \boxed{...}."

The following values were not passed to `accelerate launch` and had defaults used instead:
	`--num_machines` was set to a value of `1`
	`--mixed_precision` was set to a value of `'no'`
	`--dynamo_backend` was set to a value of `'no'`
To avoid this warning pass in values for each of the problematic parameters or run `accelerate config`.
2026-10-24:08:46:55 WARNING  [config.evaluate_config:344] --limit SHOULD ONLY BE USED FOR TESTING. REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2026-10-24:08:47:05 INFO     [_cli.run:413] Selected Tasks: ['hendrycks_math_algebra', 'hendrycks_math_counting_and_prob', 'hendrycks_math_geometry', 'hendrycks_math_intermediate_algebra', 'hendrycks_math_num_theory', 'hendrycks_math_prealgebra', 'hendrycks_math_precalc']
2026-10-24:08:47:07 INFO     [evaluator:214] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2026-10-24:08:47:07 WARNING  [evaluator:226] generation_kwargs: {'ma

In [7]:
!accelerate launch --num_processes=1 -m lm_eval --model hf \
    --model_args pretrained=arefehRajabian/deepseek_r1_finetune_16bit_persian,dtype=bfloat16 \
    --tasks hendrycks_math_algebra,hendrycks_math_counting_and_prob,hendrycks_math_geometry,hendrycks_math_intermediate_algebra,hendrycks_math_num_theory,hendrycks_math_prealgebra,hendrycks_math_precalc \
    --apply_chat_template \
    --system_instruction "Solve the following mathematics problem. Show your reasoning step by step. Put your final answer in \boxed{{...}}." \
    --gen_kwargs max_gen_toks=4096,do_sample=True,temperature=0.6,top_p=0.95 \
    --limit 10 \
    --log_samples \
    --batch_size 8 \
    --output_path /content/deepseek

The following values were not passed to `accelerate launch` and had defaults used instead:
	`--num_machines` was set to a value of `1`
	`--mixed_precision` was set to a value of `'no'`
	`--dynamo_backend` was set to a value of `'no'`
To avoid this warning pass in values for each of the problematic parameters or run `accelerate config`.
2026-10-24:10:10:41 WARNING  [config.evaluate_config:344] --limit SHOULD ONLY BE USED FOR TESTING. REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2026-10-24:10:10:41 INFO     [config.evaluate_config:364] Using default fewshot_as_multiturn=True.
2026-10-24:10:10:50 INFO     [_cli.run:413] Selected Tasks: ['hendrycks_math_algebra', 'hendrycks_math_counting_and_prob', 'hendrycks_math_geometry', 'hendrycks_math_intermediate_algebra', 'hendrycks_math_num_theory', 'hendrycks_math_prealgebra', 'hendrycks_math_precalc']
2026-10-24:10:10:52 INFO     [evaluator:214] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setti

In [4]:
!pip install -q vllm

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
grpcio-tools 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 7.36.2 which is incompatible.
ydata-profiling 4.18.4 requires numba<0.63,>=0.60, but you have numba 0.65.0 which is incompatible.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.2.2 which is incompatible.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 2.3.0 which is incompatible.
gradio 6.26.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
datasets 4.8.5 requires huggingface-hub<2.0,>=0.25.0, but you have huggingface-hub 2.3.0 which is incompatible.
pylibcudf-cu12 26.2.1 requires cuda-python<13.0,>=12.9.2, but you have cuda-python 13.4.4 which is incompatible.
numba-cuda 0.22.2 requires cuda-core<1.0.0,>=0.3.2, but you have cuda-core 1.2.2 which is inco

## **persian GRPO**

In [12]:
!python -m lm_eval --model vllm \
    --model_args pretrained=arefehRajabian/deepseek_r1_finetune_16bit_persian,dtype=bfloat16,gpu_memory_utilization=0.9,max_model_len=6144 \
    --tasks hendrycks_math_algebra hendrycks_math_counting_and_prob hendrycks_math_geometry hendrycks_math_intermediate_algebra hendrycks_math_num_theory hendrycks_math_prealgebra hendrycks_math_precalc \
    --apply_chat_template \
    --system_instruction "Solve the following mathematics problem. Show your reasoning step by step. Put your final answer in \boxed{{...}}." \
    --gen_kwargs max_gen_toks=2048,temperature=0.6,top_p=0.95 \
    --limit 100 \
    --batch_size 64 \
    --log_samples \
    --output_path /content/deepseek

2026-10-24:11:44:55 WARNING  [config.evaluate_config:344] --limit SHOULD ONLY BE USED FOR TESTING. REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2026-10-24:11:44:55 INFO     [config.evaluate_config:364] Using default fewshot_as_multiturn=True.
2026-10-24:11:45:04 INFO     [_cli.run:413] Selected Tasks: ['hendrycks_math_algebra', 'hendrycks_math_counting_and_prob', 'hendrycks_math_geometry', 'hendrycks_math_intermediate_algebra', 'hendrycks_math_num_theory', 'hendrycks_math_prealgebra', 'hendrycks_math_precalc']
2026-10-24:11:45:06 INFO     [evaluator:214] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2026-10-24:11:45:06 WARNING  [evaluator:226] generation_kwargs: {'max_gen_toks': 2048, 'temperature': 0.6, 'top_p': 0.95} specified through cli, these settings will update set parameters in yaml tasks. Ensure 'do_sample=True' for non-greedy decoding!
2026-10-24:11:45:06 INFO     [evaluator:239] Initializin

In [13]:
!python -m lm_eval --model vllm \
    --model_args pretrained=arefehRajabian/deepseek_r1_finetune_16bit_persian,dtype=bfloat16,gpu_memory_utilization=0.9,max_model_len=6144 \
    --tasks hendrycks_math_algebra hendrycks_math_counting_and_prob hendrycks_math_geometry hendrycks_math_intermediate_algebra hendrycks_math_num_theory hendrycks_math_prealgebra hendrycks_math_precalc \
    --apply_chat_template \
    --system_instruction "Solve the following mathematics problem. Show your reasoning step by step. Put your final answer in \boxed{{...}}." \
    --gen_kwargs max_gen_toks=4096,temperature=0.6,top_p=0.95 \
    --limit 100 \
    --batch_size 64 \
    --log_samples \
    --output_path /content/deepseek

2026-10-24:11:55:42 WARNING  [config.evaluate_config:344] --limit SHOULD ONLY BE USED FOR TESTING. REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2026-10-24:11:55:42 INFO     [config.evaluate_config:364] Using default fewshot_as_multiturn=True.
2026-10-24:11:55:52 INFO     [_cli.run:413] Selected Tasks: ['hendrycks_math_algebra', 'hendrycks_math_counting_and_prob', 'hendrycks_math_geometry', 'hendrycks_math_intermediate_algebra', 'hendrycks_math_num_theory', 'hendrycks_math_prealgebra', 'hendrycks_math_precalc']
2026-10-24:11:55:54 INFO     [evaluator:214] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2026-10-24:11:55:54 WARNING  [evaluator:226] generation_kwargs: {'max_gen_toks': 4096, 'temperature': 0.6, 'top_p': 0.95} specified through cli, these settings will update set parameters in yaml tasks. Ensure 'do_sample=True' for non-greedy decoding!
2026-10-24:11:55:54 INFO     [evaluator:239] Initializin

## **Ddeepseek Base**

In [14]:
!python -m lm_eval --model vllm \
    --model_args pretrained=unsloth/DeepSeek-R1-0528-Qwen3-8B,dtype=bfloat16,gpu_memory_utilization=0.9,max_model_len=6144 \
    --tasks hendrycks_math_algebra hendrycks_math_counting_and_prob hendrycks_math_geometry hendrycks_math_intermediate_algebra hendrycks_math_num_theory hendrycks_math_prealgebra hendrycks_math_precalc \
    --apply_chat_template \
    --system_instruction "Solve the following mathematics problem. Show your reasoning step by step. Put your final answer in \boxed{{...}}." \
    --gen_kwargs max_gen_toks=4096,temperature=0.6,top_p=0.95 \
    --limit 100 \
    --batch_size 64 \
    --log_samples \
    --output_path /content/deepseekBase

2026-10-24:12:27:44 WARNING  [config.evaluate_config:344] --limit SHOULD ONLY BE USED FOR TESTING. REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2026-10-24:12:27:44 INFO     [config.evaluate_config:364] Using default fewshot_as_multiturn=True.
2026-10-24:12:27:53 INFO     [_cli.run:413] Selected Tasks: ['hendrycks_math_algebra', 'hendrycks_math_counting_and_prob', 'hendrycks_math_geometry', 'hendrycks_math_intermediate_algebra', 'hendrycks_math_num_theory', 'hendrycks_math_prealgebra', 'hendrycks_math_precalc']
2026-10-24:12:27:55 INFO     [evaluator:214] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2026-10-24:12:27:55 WARNING  [evaluator:226] generation_kwargs: {'max_gen_toks': 4096, 'temperature': 0.6, 'top_p': 0.95} specified through cli, these settings will update set parameters in yaml tasks. Ensure 'do_sample=True' for non-greedy decoding!
2026-10-24:12:27:55 INFO     [evaluator:239] Initializin

## **English GRPO**

In [15]:
!python -m lm_eval --model vllm \
    --model_args pretrained=arefehRajabian/deepseek_r1_finetune_16bit,dtype=bfloat16,gpu_memory_utilization=0.9,max_model_len=6144 \
    --tasks hendrycks_math_algebra hendrycks_math_counting_and_prob hendrycks_math_geometry hendrycks_math_intermediate_algebra hendrycks_math_num_theory hendrycks_math_prealgebra hendrycks_math_precalc \
    --apply_chat_template \
    --system_instruction "Solve the following mathematics problem. Show your reasoning step by step. Put your final answer in \boxed{{...}}." \
    --gen_kwargs max_gen_toks=4096,temperature=0.6,top_p=0.95 \
    --limit 100 \
    --batch_size 64 \
    --log_samples \
    --output_path /content/deepseekBase

2026-10-24:13:03:12 WARNING  [config.evaluate_config:344] --limit SHOULD ONLY BE USED FOR TESTING. REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2026-10-24:13:03:12 INFO     [config.evaluate_config:364] Using default fewshot_as_multiturn=True.
2026-10-24:13:03:22 INFO     [_cli.run:413] Selected Tasks: ['hendrycks_math_algebra', 'hendrycks_math_counting_and_prob', 'hendrycks_math_geometry', 'hendrycks_math_intermediate_algebra', 'hendrycks_math_num_theory', 'hendrycks_math_prealgebra', 'hendrycks_math_precalc']
2026-10-24:13:03:24 INFO     [evaluator:214] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2026-10-24:13:03:24 WARNING  [evaluator:226] generation_kwargs: {'max_gen_toks': 4096, 'temperature': 0.6, 'top_p': 0.95} specified through cli, these settings will update set parameters in yaml tasks. Ensure 'do_sample=True' for non-greedy decoding!
2026-10-24:13:03:24 INFO     [evaluator:239] Initializin

In [16]:
!python -m lm_eval --model vllm \
    --model_args pretrained=arefehRajabian/deepseek_r1_finetune_16bit,dtype=bfloat16,gpu_memory_utilization=0.9,max_model_len=6144 \
    --tasks hendrycks_math_algebra hendrycks_math_counting_and_prob hendrycks_math_geometry hendrycks_math_intermediate_algebra hendrycks_math_num_theory hendrycks_math_prealgebra hendrycks_math_precalc \
    --apply_chat_template \
    --system_instruction "Solve the following mathematics problem. Show your reasoning step by step. Put your final answer in \boxed{{...}}." \
    --gen_kwargs max_gen_toks=4096,temperature=0.6,top_p=0.95 \
    --limit 100 \
    --batch_size 64 \
    --log_samples \
    --output_path /content/deepseekBase

2026-10-24:13:31:01 WARNING  [config.evaluate_config:344] --limit SHOULD ONLY BE USED FOR TESTING. REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2026-10-24:13:31:01 INFO     [config.evaluate_config:364] Using default fewshot_as_multiturn=True.
2026-10-24:13:31:10 INFO     [_cli.run:413] Selected Tasks: ['hendrycks_math_algebra', 'hendrycks_math_counting_and_prob', 'hendrycks_math_geometry', 'hendrycks_math_intermediate_algebra', 'hendrycks_math_num_theory', 'hendrycks_math_prealgebra', 'hendrycks_math_precalc']
2026-10-24:13:31:12 INFO     [evaluator:214] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2026-10-24:13:31:12 WARNING  [evaluator:226] generation_kwargs: {'max_gen_toks': 4096, 'temperature': 0.6, 'top_p': 0.95} specified through cli, these settings will update set parameters in yaml tasks. Ensure 'do_sample=True' for non-greedy decoding!
2026-10-24:13:31:12 INFO     [evaluator:239] Initializin

## **deepseek SFT**

In [1]:
!python -m lm_eval --model vllm \
    --model_args pretrained=arefe-rn/deepseek_r1_SFT_16bit,dtype=bfloat16,gpu_memory_utilization=0.9,max_model_len=6144 \
    --tasks hendrycks_math_algebra hendrycks_math_counting_and_prob hendrycks_math_geometry hendrycks_math_intermediate_algebra hendrycks_math_num_theory hendrycks_math_prealgebra hendrycks_math_precalc \
    --apply_chat_template \
    --system_instruction "Solve the following mathematics problem. Show your reasoning step by step. Put your final answer in \boxed{{...}}." \
    --gen_kwargs max_gen_toks=4096,temperature=0.6,top_p=0.95 \
    --limit 100 \
    --batch_size 64 \
    --log_samples \
    --output_path /content/deepseekBase

/usr/bin/python3: No module named lm_eval


In [8]:
!python -m lm_eval --model vllm \
    --model_args pretrained=arefe-rn/deepseek_r1_SFT_16bit,dtype=bfloat16,gpu_memory_utilization=0.9,max_model_len=6144 \
    --tasks hendrycks_math_algebra hendrycks_math_counting_and_prob hendrycks_math_geometry hendrycks_math_intermediate_algebra hendrycks_math_num_theory hendrycks_math_prealgebra hendrycks_math_precalc \
    --apply_chat_template \
    --system_instruction "Solve the following mathematics problem. Show your reasoning step by step. Put your final answer in \boxed{{...}}." \
    --gen_kwargs max_gen_toks=4096,temperature=0.6,top_p=0.95 \
    --limit 10 \
    --batch_size 64 \
    --log_samples \
    --output_path /content/deepseekSFT

2026-10-25:07:31:29 WARNING  [config.evaluate_config:344] --limit SHOULD ONLY BE USED FOR TESTING. REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2026-10-25:07:31:29 INFO     [config.evaluate_config:364] Using default fewshot_as_multiturn=True.
2026-10-25:07:31:38 INFO     [_cli.run:413] Selected Tasks: ['hendrycks_math_algebra', 'hendrycks_math_counting_and_prob', 'hendrycks_math_geometry', 'hendrycks_math_intermediate_algebra', 'hendrycks_math_num_theory', 'hendrycks_math_prealgebra', 'hendrycks_math_precalc']
2026-10-25:07:31:40 INFO     [evaluator:214] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2026-10-25:07:31:40 WARNING  [evaluator:226] generation_kwargs: {'max_gen_toks': 4096, 'temperature': 0.6, 'top_p': 0.95} specified through cli, these settings will update set parameters in yaml tasks. Ensure 'do_sample=True' for non-greedy decoding!
2026-10-25:07:31:40 INFO     [evaluator:239] Initializin

In [11]:
!python -m lm_eval --model vllm \
    --model_args pretrained=arefe-rn/deepseek_r1_SFT_16bit,tokenizer=arefehRajabian/deepseek_r1_finetune_16bit_persian,dtype=bfloat16,gpu_memory_utilization=0.9,max_model_len=6144 \
    --tasks hendrycks_math_algebra,hendrycks_math_counting_and_prob,hendrycks_math_geometry,hendrycks_math_intermediate_algebra,hendrycks_math_num_theory,hendrycks_math_prealgebra,hendrycks_math_precalc \
    --apply_chat_template \
    --system_instruction "Solve the following mathematics problem. Show your reasoning step by step. Put your final answer in \boxed{{...}}." \
    --gen_kwargs "max_gen_toks=2048,temperature=0.6,top_p=0.95,until=<|im_end|>" \
    --limit 10 \
    --log_samples \
    --output_path /content/deepseekSFT1

2026-10-25:07:47:16 WARNING  [config.evaluate_config:344] --limit SHOULD ONLY BE USED FOR TESTING. REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2026-10-25:07:47:16 INFO     [config.evaluate_config:364] Using default fewshot_as_multiturn=True.
2026-10-25:07:47:26 INFO     [_cli.run:413] Selected Tasks: ['hendrycks_math_algebra', 'hendrycks_math_counting_and_prob', 'hendrycks_math_geometry', 'hendrycks_math_intermediate_algebra', 'hendrycks_math_num_theory', 'hendrycks_math_prealgebra', 'hendrycks_math_precalc']
2026-10-25:07:47:27 INFO     [evaluator:214] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2026-10-25:07:47:27 WARNING  [evaluator:226] generation_kwargs: {'max_gen_toks': 2048, 'temperature': 0.6, 'top_p': 0.95, 'until': '<|im_end|>'} specified through cli, these settings will update set parameters in yaml tasks. Ensure 'do_sample=True' for non-greedy decoding!
2026-10-25:07:47:27 INFO     [eva

In [12]:
# p = "/content/lm-evaluation-harness/lm_eval/tasks/hendrycks_math/utils.py"
# s = open(p).read()

# helper = '''
# def _unbytelevel(text):
#     bs = list(range(ord("!"), ord("~") + 1)) + list(range(ord("\\u00a1"), ord("\\u00ac") + 1)) + list(range(ord("\\u00ae"), ord("\\u00ff") + 1))
#     cs = bs[:]
#     n = 0
#     for b in range(256):
#         if b not in bs:
#             bs.append(b)
#             cs.append(256 + n)
#             n += 1
#     inv = {chr(c): b for b, c in zip(bs, cs)}
#     out = bytearray()
#     for ch in text:
#         if ch in inv:
#             out.append(inv[ch])
#         else:
#             out.extend(ch.encode("utf-8"))
#     return out.decode("utf-8", errors="replace")

# '''
# if "_unbytelevel" not in s:
#     s = s.replace("def process_results(", helper + "def process_results(", 1)
#     s = s.replace("response = results[0]", "response = _unbytelevel(results[0]) if \"\\u0120\" in results[0] or \"\\u010a\" in results[0] else results[0]", 1)
#     open(p, "w").write(s)
# print("ok", "_unbytelevel" in open(p).read())

ok True


In [13]:
!python -m lm_eval --model vllm \
    --model_args pretrained=arefe-rn/deepseek_r1_SFT_16bit,tokenizer=arefehRajabian/deepseek_r1_finetune_16bit_persian,dtype=bfloat16,gpu_memory_utilization=0.9,max_model_len=6144 \
    --tasks hendrycks_math_algebra,hendrycks_math_counting_and_prob,hendrycks_math_geometry,hendrycks_math_intermediate_algebra,hendrycks_math_num_theory,hendrycks_math_prealgebra,hendrycks_math_precalc \
    --apply_chat_template \
    --system_instruction "Solve the following mathematics problem. Show your reasoning step by step. Put your final answer in \boxed{{...}}." \
    --gen_kwargs "max_gen_toks=2048,temperature=0.6,top_p=0.95" \
    --limit 10 \
    --log_samples \
    --output_path /content/deepseekSFT2

2026-10-25:08:42:16 WARNING  [config.evaluate_config:344] --limit SHOULD ONLY BE USED FOR TESTING. REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2026-10-25:08:42:16 INFO     [config.evaluate_config:364] Using default fewshot_as_multiturn=True.
2026-10-25:08:42:26 INFO     [_cli.run:413] Selected Tasks: ['hendrycks_math_algebra', 'hendrycks_math_counting_and_prob', 'hendrycks_math_geometry', 'hendrycks_math_intermediate_algebra', 'hendrycks_math_num_theory', 'hendrycks_math_prealgebra', 'hendrycks_math_precalc']
2026-10-25:08:42:27 INFO     [evaluator:214] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2026-10-25:08:42:27 WARNING  [evaluator:226] generation_kwargs: {'max_gen_toks': 2048, 'temperature': 0.6, 'top_p': 0.95} specified through cli, these settings will update set parameters in yaml tasks. Ensure 'do_sample=True' for non-greedy decoding!
2026-10-25:08:42:27 INFO     [evaluator:239] Initializin

In [15]:
!grep -c "_unbytelevel\|####" /content/lm-evaluation-harness/lm_eval/tasks/hendrycks_math/utils.py

0


In [16]:
   !grep -c "_unbytelevel\|####" /content/lm-evaluation-harness/lm_eval/tasks/hendrycks_math/utils.py

0


In [17]:
import json, glob, os, sys, types, importlib.util

# 1) which utils.py is on disk, and is it patched?
p = "/content/lm-evaluation-harness/lm_eval/tasks/hendrycks_math/utils.py"
print("patched:", "_unbytelevel" in open(p).read())

spec = importlib.util.spec_from_file_location("hm_utils", p)
u = importlib.util.module_from_spec(spec); spec.loader.exec_module(u)

# 2) newest run folder (change the pattern if you used another output_path)
files = sorted(glob.glob("/content/deepseekSFT*/**/samples_*.json*", recursive=True), key=os.path.getmtime)
latest_stamp = files[-1].split("_")[-1].rsplit(".", 1)[0]
files = [f for f in files if latest_stamp in f]
print(len(files), "files from run", latest_stamp)

for f in files:
    rows = [json.loads(l) for l in open(f)]
    hits, shown = 0, 0
    for r in rows:
        resp = r["resps"][0][0]
        ok = u.process_results(r["doc"], [resp])["exact_match"]
        hits += ok
        if not ok and shown < 2:
            shown += 1
            print("  FAIL doc", r["doc_id"], "| target:", r["target"])
            print("   head:", repr(resp[:150]))
            print("   tail:", repr(resp[-150:]))
    print(os.path.basename(f).split("_2026")[0], "->", hits, "/", len(rows),
          "| empty:", sum(not r["resps"][0][0].strip() for r in rows))

patched: False


/content/lm-evaluation-harness/lm_eval/tasks/hendrycks_math/utils.py:213: SyntaxWarning: invalid escape sequence '\%'
  string = string.replace("\%", "")  # noqa: W605


7 files from run 2026-10-25T09-15-05.934266
  FAIL doc 0 | target: 2
   head: '<think>ĊWeĠareĠgivenĠtheĠfunctionĠyĠ=Ġ2/(x^2+x-6)ĊWeĠwantĠtoĠfindĠtheĠverticalĠasymptotesĠofĠthisĠfunction.ĊVerticalĠasymptotesĠoccurĠwhereĠtheĠdenomi'
   tail: '####Ġ2Ċ####Ġ2<|>####Ġ2Ċ####Ġ2<|>####Ġ2Ċ####Ġ2<|>####Ġ2Ċ####Ġ2<|>####Ġ2Ċ####Ġ2<|>####Ġ2Ċ####Ġ2<|>####Ġ2Ċ####Ġ2<|>####Ġ2Ċ####Ġ2<|>####Ġ2Ċ####Ġ2<|>####Ġ2'
  FAIL doc 1 | target: 10
   head: '<think>Ċ120%ĠofĠ30ĠisĠ120/100*30=<<120/100*30=36>>36.Ċ130%ĠofĠ20ĠisĠ130/100*20=<<130/100*20=26>>26.ĊTheĠpositiveĠdifferenceĠbetweenĠ36ĠandĠ26ĠisĠ36-26'
   tail: '#Ġ10<|im_end|>Ċ####Ġ10<|im_end|>Ċ####Ġ10<|im_end|>Ċ####Ġ10<|im_end|>Ċ####Ġ10<|im_end|>Ċ####Ġ10<|im_end|>Ċ####Ġ10<|im_end|>Ċ####Ġ10<|im_end|>Ċ####Ġ10<|'
samples_hendrycks_math_algebra -> 0 / 10 | empty: 0
  FAIL doc 0 | target: \frac{10}{11}
   head: '<think>ĊTheĠprobabilityĠofĠrainĠtomorrowĠisĠ1/11,ĠsoĠtheĠprobabilityĠthatĠitĠwillĠnotĠrainĠtomorrowĠisĠ1Ġ-Ġ1/11Ġ=Ġ10/11.Ċ####Ġ10/11Ċ####Ġ10/11Ċ####Ġ10'
   tai

In [5]:
p = "/kaggle/working/lm-evaluation-harness/lm_eval/tasks/hendrycks_math/utils.py"
s = open(p).read()

new_func = '''def _unbytelevel(text):
    bs = list(range(ord("!"), ord("~") + 1)) + list(range(0xA1, 0xAC + 1)) + list(range(0xAE, 0xFF + 1))
    cs = bs[:]
    n = 0
    for b in range(256):
        if b not in bs:
            bs.append(b)
            cs.append(256 + n)
            n += 1
    inv = {chr(c): b for b, c in zip(bs, cs)}
    out = bytearray()
    for ch in text:
        if ch in inv:
            out.append(inv[ch])
        else:
            out.extend(ch.encode("utf-8"))
    return out.decode("utf-8", errors="replace")


def process_results(doc, results):
    import re as _re
    retval = 0
    response = results[0]
    if "\\u0120" in response or "\\u010a" in response:
        response = _unbytelevel(response)

    answer = None
    boxed = last_boxed_only_string(response)
    if boxed is not None:
        try:
            answer = remove_boxed(boxed)
        except AssertionError:
            answer = None

    if answer is None and "####" in response:
        answer = response.split("####")[1].split("<|")[0].strip().split("\\n")[0].strip()
        answer = _re.sub(r"<0x[0-9A-Fa-f]{2}>", "", answer).strip()

    if answer is None:
        indices = [pos for pos, char in enumerate(response) if char == "$"]
        if len(indices) <= 1:
            answer = response
        else:
            answer = response[indices[0] + 1 : indices[-1]]

    if is_equiv(answer, remove_boxed(last_boxed_only_string(doc["solution"]))):
        retval = 1
    return {"exact_match": retval}


'''

start = s.find("def process_results")
assert start != -1, "process_results not found"
end = s.find("# string normalization")
if end == -1:
    end = s.find("def is_equiv")
assert end > start, "end marker not found"
open(p, "w").write(s[:start] + new_func + s[end:])
print("patched OK")

patched OK


In [19]:
import json, glob, os, importlib.util

p = "/content/lm-evaluation-harness/lm_eval/tasks/hendrycks_math/utils.py"
print("patched:", "_unbytelevel" in open(p).read())   # must say True

spec = importlib.util.spec_from_file_location("hm_utils", p)
u = importlib.util.module_from_spec(spec); spec.loader.exec_module(u)

files = sorted(glob.glob("/content/deepseekSFT*/**/samples_*.json*", recursive=True), key=os.path.getmtime)
stamp = files[-1].split("_")[-1].rsplit(".", 1)[0]
files = [f for f in files if stamp in f]

total_hit = total = 0
for f in files:
    rows = [json.loads(l) for l in open(f)]
    hits = sum(u.process_results(r["doc"], [r["resps"][0][0]])["exact_match"] for r in rows)
    total_hit += hits; total += len(rows)
    print(os.path.basename(f).split("_2026")[0].replace("samples_hendrycks_math_", ""), f"{hits}/{len(rows)}")
print("TOTAL", f"{total_hit}/{total} = {total_hit/total:.1%}")

patched: True
algebra 3/10
counting_and_prob 4/10
geometry 0/10
intermediate_algebra 1/10
num_theory 7/10
prealgebra 3/10
precalc 1/10
TOTAL 19/70 = 27.1%


/content/lm-evaluation-harness/lm_eval/tasks/hendrycks_math/utils.py:232: SyntaxWarning: invalid escape sequence '\%'
  # remove spaces


In [20]:
!python -m lm_eval --model vllm \
    --model_args pretrained=arefe-rn/deepseek_r1_SFT_16bit,tokenizer=arefehRajabian/deepseek_r1_finetune_16bit_persian,dtype=bfloat16,gpu_memory_utilization=0.9,max_model_len=6144 \
    --tasks hendrycks_math_algebra,hendrycks_math_counting_and_prob,hendrycks_math_geometry,hendrycks_math_intermediate_algebra,hendrycks_math_num_theory,hendrycks_math_prealgebra,hendrycks_math_precalc \
    --apply_chat_template \
    --system_instruction "Solve the following mathematics problem. Show your reasoning step by step. Put your final answer in \boxed{{...}}." \
    --gen_kwargs "max_gen_toks=4096,temperature=0.6,top_p=0.95" \
    --limit 10 \
    --log_samples \
    --output_path /content/deepseekSFT5

2026-10-25:09:45:42 WARNING  [config.evaluate_config:344] --limit SHOULD ONLY BE USED FOR TESTING. REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2026-10-25:09:45:42 INFO     [config.evaluate_config:364] Using default fewshot_as_multiturn=True.
2026-10-25:09:45:52 INFO     [_cli.run:413] Selected Tasks: ['hendrycks_math_algebra', 'hendrycks_math_counting_and_prob', 'hendrycks_math_geometry', 'hendrycks_math_intermediate_algebra', 'hendrycks_math_num_theory', 'hendrycks_math_prealgebra', 'hendrycks_math_precalc']
2026-10-25:09:45:53 INFO     [evaluator:214] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2026-10-25:09:45:53 WARNING  [evaluator:226] generation_kwargs: {'max_gen_toks': 4096, 'temperature': 0.6, 'top_p': 0.95} specified through cli, these settings will update set parameters in yaml tasks. Ensure 'do_sample=True' for non-greedy decoding!
2026-10-25:09:45:53 INFO     [evaluator:239] Initializin

In [ ]:
!python -m lm_eval --model vllm \
    --model_args pretrained=arefe-rn/deepseek_r1_SFT_16bit,tokenizer=arefehRajabian/deepseek_r1_finetune_16bit_persian,dtype=bfloat16,gpu_memory_utilization=0.9,max_model_len=6144 \
    --tasks hendrycks_math_algebra,hendrycks_math_counting_and_prob,hendrycks_math_geometry,hendrycks_math_intermediate_algebra,hendrycks_math_num_theory,hendrycks_math_prealgebra,hendrycks_math_precalc \
    --apply_chat_template \
    --system_instruction "Solve the following mathematics problem. Show your reasoning step by step. Put your final answer in \boxed{{...}}." \
    --gen_kwargs "max_gen_toks=4096,temperature=0.6,top_p=0.95" \
    --limit 100 \
    --batch_size auto \
    --log_samples \
    --output_path /content/deepseekSFT6

2026-10-25:10:54:44 WARNING  [config.evaluate_config:344] --limit SHOULD ONLY BE USED FOR TESTING. REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2026-10-25:10:54:44 INFO     [config.evaluate_config:364] Using default fewshot_as_multiturn=True.
2026-10-25:10:54:54 INFO     [_cli.run:413] Selected Tasks: ['hendrycks_math_algebra', 'hendrycks_math_counting_and_prob', 'hendrycks_math_geometry', 'hendrycks_math_intermediate_algebra', 'hendrycks_math_num_theory', 'hendrycks_math_prealgebra', 'hendrycks_math_precalc']
2026-10-25:10:54:56 INFO     [evaluator:214] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2026-10-25:10:54:56 WARNING  [evaluator:226] generation_kwargs: {'max_gen_toks': 4096, 'temperature': 0.6, 'top_p': 0.95} specified through cli, these settings will update set parameters in yaml tasks. Ensure 'do_sample=True' for non-greedy decoding!
2026-10-25:10:54:56 INFO     [evaluator:239] Initializin

In [7]:
!pip uninstall -y torchaudio

Found existing installation: torchaudio 2.11.0+cu128
Uninstalling torchaudio-2.11.0+cu128:
  Successfully uninstalled torchaudio-2.11.0+cu128


In [8]:
# !python -m lm_eval --model vllm \
#     --model_args pretrained=arefe-rn/deepseek_r1_SFT_16bit,tokenizer=arefehRajabian/deepseek_r1_finetune_16bit_persian,dtype=bfloat16,gpu_memory_utilization=0.9,max_model_len=6144 \
#     --tasks hendrycks_math_algebra,hendrycks_math_counting_and_prob,hendrycks_math_geometry,hendrycks_math_intermediate_algebra,hendrycks_math_num_theory,hendrycks_math_prealgebra,hendrycks_math_precalc \
#     --apply_chat_template \
#     --system_instruction "Solve the following mathematics problem. Show your reasoning step by step. Put your final answer in \boxed{{...}}." \
#     --gen_kwargs "max_gen_toks=4096,temperature=0.6,top_p=0.95" \
#     --limit 100 \
#     --batch_size auto \
#     --log_samples \
#     --output_path /kaggle/working/deepseekSFT6

!python -m lm_eval --model vllm \
    --model_args pretrained=arefe-rn/deepseek_r1_SFT_16bit,tokenizer=arefehRajabian/deepseek_r1_finetune_16bit_persian,dtype=float16,tensor_parallel_size=2,gpu_memory_utilization=0.9,max_model_len=4096,enforce_eager=True \
    --tasks hendrycks_math_algebra,hendrycks_math_counting_and_prob,hendrycks_math_geometry,hendrycks_math_intermediate_algebra,hendrycks_math_num_theory,hendrycks_math_prealgebra,hendrycks_math_precalc \
    --apply_chat_template \
    --system_instruction "Solve the following mathematics problem. Show your reasoning step by step. Put your final answer in \boxed{{...}}." \
    --gen_kwargs "max_gen_toks=2048,temperature=0.6,top_p=0.95,until=<|im_end|>" \
    --batch_size auto \
    --limit 10 \
    --log_samples \
    --output_path /kaggle/working/deepseekSFT

2026-10-25:12:28:24 WARNING  [config.evaluate_config:344] --limit SHOULD ONLY BE USED FOR TESTING. REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2026-10-25:12:28:24 INFO     [config.evaluate_config:364] Using default fewshot_as_multiturn=True.
2026-10-25:12:28:34 INFO     [_cli.run:413] Selected Tasks: ['hendrycks_math_algebra', 'hendrycks_math_counting_and_prob', 'hendrycks_math_geometry', 'hendrycks_math_intermediate_algebra', 'hendrycks_math_num_theory', 'hendrycks_math_prealgebra', 'hendrycks_math_precalc']
2026-10-25:12:28:36 INFO     [evaluator:214] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2026-10-25:12:28:36 WARNING  [evaluator:226] generation_kwargs: {'max_gen_toks': 2048, 'temperature': 0.6, 'top_p': 0.95, 'until': '<|im_end|>'} specified through cli, these settings will update set parameters in yaml tasks. Ensure 'do_sample=True' for non-greedy decoding!
2026-10-25:12:28:36 INFO     [eva

In [9]:
!python -m lm_eval --model vllm \
    --model_args pretrained=arefe-rn/deepseek_r1_SFT_16bit,tokenizer=arefehRajabian/deepseek_r1_finetune_16bit_persian,dtype=float16,tensor_parallel_size=2,gpu_memory_utilization=0.9,max_model_len=4096,enforce_eager=True \
    --tasks hendrycks_math_algebra,hendrycks_math_counting_and_prob,hendrycks_math_geometry,hendrycks_math_intermediate_algebra,hendrycks_math_num_theory,hendrycks_math_prealgebra,hendrycks_math_precalc \
    --apply_chat_template \
    --system_instruction "Solve the following mathematics problem. Show your reasoning step by step. Put your final answer in \boxed{{...}}." \
    --gen_kwargs "max_gen_toks=2048,temperature=0.6,top_p=0.95,until=<|im_end|>" \
    --batch_size auto \
    --limit 100 \
    --log_samples \
    --output_path /kaggle/working/deepseekSFT

2026-10-25:12:41:40 WARNING  [config.evaluate_config:344] --limit SHOULD ONLY BE USED FOR TESTING. REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2026-10-25:12:41:40 INFO     [config.evaluate_config:364] Using default fewshot_as_multiturn=True.
2026-10-25:12:41:51 INFO     [_cli.run:413] Selected Tasks: ['hendrycks_math_algebra', 'hendrycks_math_counting_and_prob', 'hendrycks_math_geometry', 'hendrycks_math_intermediate_algebra', 'hendrycks_math_num_theory', 'hendrycks_math_prealgebra', 'hendrycks_math_precalc']
2026-10-25:12:41:53 INFO     [evaluator:214] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2026-10-25:12:41:53 WARNING  [evaluator:226] generation_kwargs: {'max_gen_toks': 2048, 'temperature': 0.6, 'top_p': 0.95, 'until': '<|im_end|>'} specified through cli, these settings will update set parameters in yaml tasks. Ensure 'do_sample=True' for non-greedy decoding!
2026-10-25:12:41:53 INFO     [eva